# Anforderungen installieren

In [7]:
pip install -r requirements.txt

/bin/bash: line 1: /workspaces/suva_challeng2025_new/.venv/bin/python: No such file or directory
Note: you may need to restart the kernel to use updated packages.


# Setup & Imports

In [3]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime
import requests

# Verzeichnisse
DATA_DIR = "./data"
REPORT_DIR = "./reports"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

today = datetime.now().strftime("%Y-%m-%d")
CSV_PATH = os.path.join(DATA_DIR, f"egid_buildings_{today}.csv")
REPORT_PATH = os.path.join(REPORT_DIR, f"updatereport_{today}.txt")


# Datenquelle & Download

In [4]:
url = (
    "https://data.egid.ch/current?"
    "format=csv&"
    "sql=SELECT+*+FROM+building+WHERE+GBAUJ+<=+1990+ORDER+BY+GBAUJ"
)

print("🌐 Lade CSV von data.egid.ch ...")
response = requests.get(url)
response.raise_for_status()

with open(CSV_PATH, "wb") as f:
    f.write(response.content)

print(f"✅ CSV gespeichert unter {CSV_PATH}")


🌐 Lade CSV von data.egid.ch ...
✅ CSV gespeichert unter ./data/egid_buildings_2025-10-18.csv


# CSV File einlesen

In [5]:
### Schrit 3 muss überarbeitet werden mit ChatGPT, da kein Dropna bei Leeren feldern.

In [6]:
# === CSV einlesen & bereinigen ===
df = pd.read_csv(CSV_PATH)

# Spalten umbenennen für Konsistenz
rename_map = {
    "EGID": "egid",
    "GGDENAME": "gemeinde",
    "GDEKT": "kanton",
    "GBAUJ": "baujahr",
    "GKODN": "lat",
    "GKODE": "lon",
    "GKAT": "gebaeudekategorie"
}
df = df.rename(columns=rename_map)

# Nur Datensätze ohne EGID (Pflichtfeld) rauswerfen
df = df[df["egid"].notna()]

# Datentypen vereinheitlichen
df["egid"] = df["egid"].astype(int)
df["baujahr"] = df["baujahr"].fillna(0).astype(int)

# Leere Felder bei lat/lon oder anderen Spalten bleiben bewusst erhalten
# → Enduser können diese Daten später ergänzen

# Standardwerte hinzufügen
df["asbestrisiko"] = 0
df["gebaeudestatus"] = 0
df["erstellt_von"] = "csv_import"
df["geaendert_von"] = "csv_import"

print(f"📊 {len(df)} Datensätze für den Import vorbereitet (inkl. leerer Felder).")
df.head(10)


ParserError: Error tokenizing data. C error: Expected 1 fields in line 6, saw 3


# Verbindung zur PostgreSQL-Datenbank

In [ ]:
Schritt 4 überarbeiten, die variblen sind im .env und dürfen nicht hier eingetragen werden wegen github pubilikation

# Zweck von Schritt 5 kurz erklären lassen bevor umsetzen

# Updatereport erzeugen

In [ ]:
cur.execute("""
SELECT egid, gemeinde, kanton
FROM allegebaeude
WHERE geaendert_von = 'user';
""")

user_locked = cur.fetchall()

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(f"Update-Report für {today}\n")
    f.write("=" * 60 + "\n\n")
    f.write("Gebäude, die wegen Useränderungen NICHT überschrieben wurden:\n\n")
    if user_locked:
        for row in user_locked:
            f.write(f"EGID {row[0]} – {row[1]}, {row[2]}\n")
    else:
        f.write("Keine User-geschützten Datensätze gefunden.\n")

print(f"📄 Report erstellt: {REPORT_PATH}")

cur.close()
conn.close()
